In [1]:
from datasets import load_dataset, ClassLabel, concatenate_datasets
from transformers import AutoTokenizer, DataCollatorForLanguageModeling, DataCollatorWithPadding
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer, AutoModelForSequenceClassification
from transformers import Trainer
from evaluate import load
import bitsandbytes as bnb

In [2]:
# - for venv
# python3.11 -m venv py311
# - for conda
# conda create -n py311 python=3.11

# !pip install bitsandbytes evaluate

In [3]:
from huggingface_hub import login

login()

In [4]:
ds1 = load_dataset('benjaminvdb/dbrd')
ds2 = load_dataset('yhavinga/imdb_dutch', revision='3121d57e4f57cc71f3aec1fc4fd9c7666b95be63')

In [5]:
ds3 = concatenate_datasets([ds1['train'], ds2['train']])
ds4 = concatenate_datasets([ds1['test'], ds2['test']])

In [6]:
# model_name = "FacebookAI/xlm-roberta-base"
model_name = 'FacebookAI/xlm-roberta-large'
model_name = "DTAI-KULeuven/robbert-2023-dutch-large"
# dataset_name = "benjaminvdb/dbrd"
# dataset_name = 'corona-tweet/dutch_social'
model = AutoModelForSequenceClassification.from_pretrained(model_name) #VOOR DUTCH_SOCIAL NUM_LABELS OP 3 PROBEREN!!
tokenizer = AutoTokenizer.from_pretrained(model_name)
# dataset = load_dataset('benjaminvdb/dbrd')
# dataset = load_dataset(dataset_name, revision='4249d01738f3e2d2944dd2d5daf4283f8f7a6ce1')
# dataset = load_dataset('benjaminvdb/dbrd')

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: DTAI-KULeuven/robbert-2023-dutch-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
# # dataset["validation"].features
# new_train_features = dataset["train"].features.copy()
# new_train_features['label'] = ClassLabel(num_classes=3, names=['negative', 'neutral', 'positive'])
# new_test_features = dataset["test"].features.copy()
# new_test_features['label'] = ClassLabel(num_classes=3, names=['negative', 'neutral', 'positive'])
# new_val_features = dataset["validation"].features.copy()
# new_val_features['label'] = ClassLabel(num_classes=3, names=['negative', 'neutral', 'positive'])

In [8]:
# # dataset["train"].features
# dataset = dataset.filter(lambda x: x['label'] != 1)
# dataset = dataset.cast_column('label', ClassLabel(names=['negative', 'neutral', 'positive']))
# # def map_labels(sample):
# #     label = sample['label']
# #     if label == 2 | label == 'pos':
# #         sample['label'] = 1
# #     return sample

# # dataset = dataset.map(map_labels)
# # dataset

In [9]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        max_length=256,
        padding="max_length",
        truncation=True,
    )

# def tokenize(examples):
#     return tokenizer(examples["text"], padding="max_length", truncation=True)

# ds1 = ds1.map(tokenize, batched=True)
ds3 = ds3.map(tokenize, batched=True)
ds4 = ds4.map(tokenize, batched=True)
# dataset = dataset.map(tokenize, batched=True)

In [10]:
# # dataset["train"].features
# dataset = dataset.filter(lambda x: x['label'] != 1)
# dataset = dataset.cast_column('label', ClassLabel(num_classes=2, names=['neg', 'pos']))
# dataset

In [11]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # convert the logits to their predicted class
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [12]:
# training_args = TrainingArguments(
#     output_dir="roberta-finetuned-2",
#     eval_strategy="epoch",
#     push_to_hub=True,
# )

training_args = TrainingArguments(
    # output_dir=f"{str(dataset_name).replace('/', '-')}",           # Directory for saving model checkpoints
    output_dir="xlm-roberta-large-with-dbrb-pc",
    eval_strategy="epoch",     # Evaluate at the end of each epoch
    save_strategy="epoch",
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    torch_compile=True,
    optim="adamw_bnb_8bit",
    learning_rate=5e-5,              # Start with a small learning rate
    per_device_train_batch_size=4,  # Batch size per GPU
    per_device_eval_batch_size=4,
    num_train_epochs=3,              # Number of epochs
    weight_decay=0.01,               # Regularization
    save_total_limit=2,              # Limit checkpoints to save space
    load_best_model_at_end=True,     # Automatically load the best checkpoint
    logging_dir="./logs",            # Directory for logs
    logging_steps=100,               # Log every 100 steps
    fp16=True                        # Enable mixed precision for faster training
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [13]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,                        # Pre-trained BERT model
    args=training_args,                 # Training arguments
    train_dataset=ds3,
    eval_dataset=ds4,
    processing_class=tokenizer,
    data_collator=data_collator,        # Efficient batching
    compute_metrics=compute_metrics     # Custom metric
)

In [14]:
trainer.train()

trainer.push_to_hub()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 3}.
W0331 23:44:50.395000 49947 site-packages/torch/_inductor/utils.py:1613] [0/0] Not enough SMs to use max_autotune_gemm mode
/home/luukv/anaconda3/lib/python3.13/site-packages/torch/_inductor/lowering.py:7242: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
/home/luukv/anaconda3/lib/python3.13/site-packages/torch/_inductor/lowering.py:7242: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy
1,2.807948,0.693366,0.499853
2,2.787741,0.696752,0.500147
3,2.781073,0.693114,0.500147


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/sytnaxerror/xlm-roberta-large-with-dbrb-pc/commit/334a745d723adcfcf73f77bfa46d5ad135d3b7fe', commit_message='End of training', commit_description='', oid='334a745d723adcfcf73f77bfa46d5ad135d3b7fe', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sytnaxerror/xlm-roberta-large-with-dbrb-pc', endpoint='https://huggingface.co', repo_type='model', repo_id='sytnaxerror/xlm-roberta-large-with-dbrb-pc'), pr_revision=None, pr_num=None)

In [15]:
# model = AutoModelForSequenceClassification.from_pretrained('DTAI-KULeuven/robbert-2023-dutch-large') #VOOR DUTCH_SOCIAL NUM_LABELS OP 3 PROBEREN!!
# training_args = TrainingArguments(
#     # output_dir=f"{str(dataset_name).replace('/', '-')}",           # Directory for saving model checkpoints
#     output_dir="corona-tweet-dutch_social-test5",
#     eval_strategy="epoch",     # Evaluate at the end of each epoch
#     save_strategy="epoch",
#     learning_rate=5e-5,              # Start with a small learning rate
#     per_device_train_batch_size=16,  # Batch size per GPU
#     per_device_eval_batch_size=16,
#     num_train_epochs=3,              # Number of epochs
#     weight_decay=0.01,               # Regularization
#     save_total_limit=2,              # Limit checkpoints to save space
#     load_best_model_at_end=True,     # Automatically load the best checkpoint
#     logging_dir="./logs",            # Directory for logs
#     logging_steps=100,               # Log every 100 steps
#     fp16=True                        # Enable mixed precision for faster training
# )
# trainer.train()

# trainer.push_to_hub()

# model = AutoModelForSequenceClassification.from_pretrained('DTAI-KULeuven/robbert-2023-dutch-base') #VOOR DUTCH_SOCIAL NUM_LABELS OP 3 PROBEREN!!
# training_args = TrainingArguments(
#     # output_dir=f"{str(dataset_name).replace('/', '-')}",           # Directory for saving model checkpoints
#     output_dir="corona-tweet-dutch_social-test6",
#     eval_strategy="epoch",     # Evaluate at the end of each epoch
#     save_strategy="epoch",
#     learning_rate=5e-5,              # Start with a small learning rate
#     per_device_train_batch_size=16,  # Batch size per GPU
#     per_device_eval_batch_size=16,
#     num_train_epochs=3,              # Number of epochs
#     weight_decay=0.01,               # Regularization
#     save_total_limit=2,              # Limit checkpoints to save space
#     load_best_model_at_end=True,     # Automatically load the best checkpoint
#     logging_dir="./logs",            # Directory for logs
#     logging_steps=100,               # Log every 100 steps
#     fp16=True                        # Enable mixed precision for faster training
# )
# trainer.train()

# trainer.push_to_hub()